In [ ]:
import pandas as pd

In [ ]:
strict = pd.read_csv('RawTotalCites(strict).csv', index_col=0, names=['Paragraph','Strict Score'])

In [ ]:
strict

In [ ]:
agressive = pd.read_csv('RawTotalCites.csv', index_col=0, names=['Paragraph','Aggressive Score'])

In [ ]:
agressive

In [ ]:
strict['Agressive Score'] = agressive['Aggressive Score']

In [ ]:
strict['Agressive Score']['Abs35']

In [ ]:
def df_col_similarity(df, colx, coly):
    total_diff = 0
    for para in df.index:
        total_diff += abs(df[colx][para] - df[coly][para])
    return total_diff

In [ ]:
df_col_similarity(strict, 'Agressive Score', 'Strict Score')

In [ ]:
strict['Difference'] = abs(strict['Agressive Score'] - strict['Strict Score'])

In [ ]:
strict.sort_values('Difference', ascending=False).to_csv('Strict_Agg_comp.csv')

In [ ]:
strict_comp_data = pd.read_csv("overall_similarity_scores(strict).csv")
aggressive_comp_data = pd.read_csv("overall_similarity_scores.csv")

In [ ]:
strict_comp_data['RelativeTotalCites'].sort_values().plot.bar()

In [ ]:
aggressive_comp_data['RelativeTotalCites'].sort_values().plot.bar()

In [ ]:
strict_cite_counter = pd.read_csv("cite_counts(strict).csv", index_col=0, names=['Counts'])
agg_cite_counter = pd.read_csv("cite_counts.csv", index_col=0, names=['Counts'])

In [ ]:
strict_cite_counter.columns

In [ ]:
strict_cite_counter['Counts'].sort_values().plot.bar()

In [ ]:
agg_cite_counter['Counts'].sort_values().plot.bar()

In [ ]:
articles = 'Unnamed: 0'

In [ ]:
aggressive_comp_data[articles]

In [ ]:
art_names = aggressive_comp_data[articles]
art_names

In [ ]:
a = aggressive_comp_data[art_names[70]] < 102
b = aggressive_comp_data[art_names[70]] > 98

In [ ]:
drop_index = aggressive_comp_data[art_names[70]][a & b].index
dropper = drop_index.values.tolist()
aggressive_comp_data.drop(drop_index).sort_values(art_names[70])[art_names[70]]

In [ ]:
def get_relative_to_lit(art_name):
    article_filter = aggressive_comp_data[articles] == art_name
    score = aggressive_comp_data[article_filter]['RelativeTotalCites'].values[0]
    return score

In [ ]:
def most_similar(article):
    #exclude no citeswhich are all about 100
    no_cites_filter = (aggressive_comp_data[article] > 99) & (aggressive_comp_data[article] < 101)
    excluded_no_cites = aggressive_comp_data[no_cites_filter]
        
    #generate list of top 10 and those within 5% of top 10
    top_10 = excluded_no_cites[article].sort_values()[1:11]
    within_5_of_num_ten = []
    num_10_name = excluded_no_cites[article].sort_values().tolist()[11]
    g_no_10_filter = excluded_no_cites[article] > no_10_score

In [ ]:
aggressive_comp_data['project_muse_511194'].sort_values()

In [ ]:
aggressive_comp_data[(aggressive_comp_data['project_muse_511194'] > 99) & (aggressive_comp_data['project_muse_511194'] < 101)]

In [ ]:
test_score = aggressive_comp_data[aggressive_comp_data[articles]=='project_muse_511194']['RelativeTotalCites']
test_score

In [ ]:
more_than_5_lower = aggressive_comp_data['RelativeTotalCites'] > (.95 * test_score.values[0])
more_than_5_lower

In [ ]:
less_than_five_higher = aggressive_comp_data['RelativeTotalCites'] < (1.05 * test_score.values[0])
less_than_five_higher

In [ ]:
five_list = aggressive_comp_data[less_than_five_higher & more_than_5_lower][articles]
five_list.sort_values()

In [ ]:
art_list = aggressive_comp_data.sort_values('project_muse_511194')[articles][0:len(five_list)]
tenth_article = art_list.values.tolist()[11]
print(tenth_article)
tenth_article_score = aggressive_comp_data[aggressive_comp_data[articles] == tenth_article]['project_muse_511194'].values[0]
print(tenth_article_score)
within_5 = (aggressive_comp_data['project_muse_511194'] < (tenth_article_score * 1.05)) & (aggressive_comp_data['project_muse_511194'] > tenth_article_score)
articles_within_5 = aggressive_comp_data[within_5]
articles_within_5

OKAY so the following function does a good job returning the top 10 most simliar articlse along with any articles that are within 5% of the similarity score of the 10th article

In [ ]:
def most_similar(article):
    #exclude no citeswhich are all about 100 for relative total cites
    #since that's the difference betwen total cites and no cits
    #it would be better to just filter them out from the start though so we don't have to worry about dealing wiht the
    no_cites_filter = (aggressive_comp_data['RelativeTotalCites'] > 99) & (aggressive_comp_data['RelativeTotalCites'] < 101)
    cites_to_exclude = aggressive_comp_data[no_cites_filter]
    excluded_cites = aggressive_comp_data.drop(cites_to_exclude.index)
        
    #generate list of top 10 and those within 5% of top 10
    top_10 = excluded_cites.sort_values(article)[1:11]
    num_10_name = top_10[articles].values.tolist()[9]
    num_10_score = excluded_cites[excluded_cites[articles] == num_10_name][article].values[0]
    within_5_of_num_10 = excluded_cites[article] <= 1.05 * num_10_score
    result_articles = excluded_cites[within_5_of_num_10].sort_values(article)[[articles,article]][1:]
    
    return result_articles.values

In [ ]:
most_similar('project_muse_383238')

The next thing to work on is getting the percentiles of similarity scores so we can grade match scores. IE 90th percentile or better mathes = A quality match. 80th percentile is a B quality Match, etc.

To Start, let's work with all the papers that actually exist

In [ ]:
no_cites_filter = (aggressive_comp_data['RelativeTotalCites'] > 99) & (aggressive_comp_data['RelativeTotalCites'] < 101)
cites_to_exclude = aggressive_comp_data[no_cites_filter]
excluded_cites = aggressive_comp_data.drop(cites_to_exclude.index)
excluded_cites

In [ ]:
numbers_only = excluded_cites.drop(axis=1, labels=articles)

In [ ]:
numbers_only

In [ ]:
all_scores = pd.Series(dtype='float64')

In [ ]:
for column in numbers_only.columns:
    all_scores = all_scores.append(numbers_only[column])

In [ ]:
ranks = all_scores[all_scores>0].rank(pct=True)
#this version has no zeros
all_scores[all_scores>0].describe()

In [ ]:
#this is the total body
all_scores.describe()

In [ ]:
for mark in [.1, .2, .3, .4, .5, .6, .7, .8, .9]:
    print(all_scores[all_scores>0].quantile(q=mark))

In [ ]:
similarity_grade_dict = {}
similarity_grade_dict['A'] = all_scores[all_scores>0].quantile(q=.1)
similarity_grade_dict['B'] = all_scores[all_scores>0].quantile(q=.2)
similarity_grade_dict['C'] = all_scores[all_scores>0].quantile(q=.3)
similarity_grade_dict['D'] = all_scores[all_scores>0].quantile(q=.4)

In [ ]:
def similarity_grader(score):
    if score < similarity_grade_dict['A']:
        return 'A'
    elif score < similarity_grade_dict['B']:
        return 'B'
    elif score < similarity_grade_dict['C']:
        return 'C'
    elif score < similarity_grade_dict['D']:
        return 'D'
    else:
        return 'F'

In [ ]:
similarity_grader(99)

now that I have a grader for the similarity scores, we can attach a grade to each of the similarity scores we find in the similarity finder. Or I could use these grades to filter the similarity grades. Ie Only return C or better. 

Or we could only look for articles with or without a certain number of A or better than B, etc. quality matches.

It would also be good to filter by number of cites in the article, etc.

Below is a way to locate the percentile rank for a given score

In [ ]:
non_zeroes = all_scores[all_scores>0]

In [ ]:
scores_w_rank = pd.DataFrame(non_zeroes, columns=['scores'])

In [ ]:
scores_w_rank['Ranks'] = round(scores_w_rank['scores'].rank(pct=True)*100, 1)

In [ ]:
scores_w_rank.sort_values('scores')